# Whole-Genome Germline Variant Analysis Pipeline

## Germline Variant Calling Pipeline from BAM Files

### Workflow Overview

1. Download and Install Required Tools
2. Upload Input BAM Files
3. Download and Prepare the Reference Genome
4. Add Read Groups
5. Mark PCR Duplicates
6. Base Quality Score Recalibration (BQSR)
7. Germline Variant Discovery with GATK HaplotypeCaller
8. Extract SNPs and INDELs
9. Variant Filtration
10. Functional Annotation with SnpEff
11. Review Final Results

### Download and Install GATK (Genome Analysis Toolkit)

In [ ]:
# 1. Download the latest stable GATK release (v4.6.2.0)
!wget https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

# 2. Unzip the package quietly
!unzip -q gatk-4.6.2.0.zip

# 3. Add the GATK directory to the system PATH so you can call 'gatk' directly from any cell
import os
os.environ['PATH'] += ':_ROOT_/content/gatk-4.6.2.0'
# For standard Colab environments, the absolute path is /content/gatk-4.6.2.0
os.environ['PATH'] += ':/content/gatk-4.6.2.0'

In [ ]:
# 4. Verify installation
!gatk --version

### Install SAMtools for BAM File Processing

In [ ]:
!apt-get install -y samtools

## Uploading and unziping Data (BAM file)

In [ ]:
%%bash
# Use standard unzip to extract the files cleanly
unzip -qo /content/WES-LUNG.zip -d /content/

# Verify that the files are now successfully sitting in /content/WES-LUNG/
echo "📊 Checking extracted files:"
ls -lh /content/WES-LUNG/

## Step 1: Download and Index Reference

In [ ]:
%%bash
mkdir -p /content/reference_hg19
echo "📥 Downloading matching UCSC hg19 reference..."

# Download the hg19 fasta file
wget -q --show-progress -P /content/reference_hg19/ https://hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz
gunzip /content/reference_hg19/hg19.fa.gz

# Index the fasta file
echo "🔧 Indexing reference..."
samtools faidx /content/reference_hg19/hg19.fa

# Create the sequence dictionary
echo "🔧 Creating sequence dictionary..."
./gatk-4.6.2.0/gatk CreateSequenceDictionary -R /content/reference_hg19/hg19.fa

## Adding Read Groups and Marking Duplicates
- Read Groups (@RG tags) identify the sample name, sequencing platform, and library. We will use GATK's AddOrReplaceReadGroups and sort the file by genomic coordinates.

- During PCR amplification in sequencing, the exact same DNA fragment can be sequenced multiple times. We need to flag these "artifacts" so they don't skew our variant calling statistics.

In [ ]:
%%bash
# --- 1. PROCESS THE NORMAL SAMPLE ---
echo "🔧 Preprocessing NORMAL tissue..."
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/NORMAL.bam \
    -O /content/NORMAL.rg.bam \
    -RGID 1 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit1 -RGSM NORMAL \
    --CREATE_INDEX true

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/NORMAL.rg.bam \
    -O /content/NORMAL.marked_dups.bam \
    -M /content/normal_metrics.txt \
    --CREATE_INDEX true

# --- 2. PROCESS THE TUMOR SAMPLE ---
echo "🔧 Preprocessing TUMOR tissue..."
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/TUMOR.bam \
    -O /content/TUMOR.rg.bam \
    -RGID 2 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit2 -RGSM TUMOR \
    --CREATE_INDEX true

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/TUMOR.rg.bam \
    -O /content/TUMOR.marked_dups.bam \
    -M /content/tumor_metrics.txt \
    --CREATE_INDEX true

echo "✅ Preprocessing complete! Both BAMs are ready."

### Step 3: Base Quality Score Recalibration (BQSR)
The sequencing machine often introduces systematic errors when assigning quality scores to bases. BQSR uses a database of known polymorphic sites (like dbSNP) to adjust these quality scores so they reflect the true error probability.

⚠️ As my data is 0.1% subsample, BQSR might struggle or throw warnings due to low data volume, For BQSR, we need to run it like this:

In [ ]:
# 1. Download known sites (dbSNP for hg19)
# !wget ftp://ftp.ncbi.nih.gov/snp/organisms/human_9606_b151_GRCh37p13/VCF/All_20180418.vcf.gz

# 2. Build the recalibration table
# !./gatk-4.6.2.0/gatk BaseRecalibrator \
 #    -I /content/WES-LUNG_dedup.bam \
  #   -R hg19.fa \
   #  --known-sites All_20180418.vcf.gz \
    # -O /content/recal_data.table

# 3. Apply the recalibration to the BAM
# !./gatk-4.6.2.0/gatk ApplyBQSR \
  #   -I /content/WES-LUNG_dedup.bam \
   #  -R hg19.fa \
    # --bqsr-recal-file /content/recal_data.table \
    # -O /content/WES-LUNG_final.bam

## Germline Variant Discovery

In [ ]:
!java -Xmx4g -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar HaplotypeCaller \
    -R /content/reference_hg19/hg19.fa \
    -I /content/NORMAL.marked_dups.bam \
    -O /content/germline_raw.vcf

## Variant Filtration

In [ ]:
%%bash
echo "🧹 Applying relaxed filters for downsampled data..."
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar VariantFiltration \
    -R /content/reference_hg19/hg19.fa \
    -V /content/germline_raw.vcf \
    -filter "QUAL < 10.0" --filter-name "LowQual" \
    -O /content/germline_filtered_relaxed.vcf

# Isolate header metadata and passing variants
grep -E '^#|PASS' /content/germline_filtered_relaxed.vcf > /content/germline_final_passed.vcf

echo "📊 High-Confidence Germline Variants Remaining:"
grep -v '^#' /content/germline_final_passed.vcf | wc -l

## Functional Annotation via SnpEff

In [ ]:
# Install and Setup SnpEff

%%bash
cd /content/
wget -q --show-progress https://downloads.sourceforge.net/project/snpeff/snpEff_latest_core.zip
unzip -qo snpEff_latest_core.zip
echo "📊 Verifying jar placement:"
ls -lh /content/snpEff/snpEff.jar

In [ ]:
# Functional Annotation of Variants Using SnpEff

!java -Xmx4g -jar /content/snpEff/snpEff.jar \
    hg19 \
    /content/germline_final_passed.vcf \
    > /content/germline_annotated.vcf

## Inspect Variants

In [ ]:
import pandas as pd

vcf_path = "/content/germline_annotated.vcf"
germline_variants = []

with open(vcf_path, 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        chunks = line.strip().split('\t')
        info = chunks[7]

        if "ANN=" in info:
            ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
            first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')

            gene_name = first_effect[3]
            effect = first_effect[1]
            impact = first_effect[2]

            germline_variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])

df_germline = pd.DataFrame(germline_variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

print("=== COMPLETE GERMLINE VARIANT PROFILE (ALL IMPACT LEVELS) ===")
if not df_germline.empty:
    # Display the top 20 variants to see what HaplotypeCaller captured
    print(df_germline.head(20).to_string(index=False))
    print(f"\n📊 Total background variants found in this slice: {len(df_germline)}")
else:
    print("The variant file is completely empty. Double-check if 'germline_final_passed.vcf' contains variants.")

## Extract SNP and INDELS

In [ ]:
%%bash
# 1. Extract only the Single Nucleotide Polymorphisms (SNPs)
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/germline_final_passed.vcf \
    -select-type SNP \
    -O /content/germline_snps.vcf

# 2. Extract only the Insertions and Deletions (Indels)
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/germline_final_passed.vcf \
    -select-type INDEL \
    -O /content/germline_indels.vcf

echo "📊 Quick Count Breakdown:"
echo -n "Total SNPs: " && grep -v '^#' /content/germline_snps.vcf | wc -l
echo -n "Total Indels: " && grep -v '^#' /content/germline_indels.vcf | wc -l

In [ ]:
# Annotate SNPs
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/germline_snps.vcf > /content/germline_snps_annotated.vcf

# Annotate Indels
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/germline_indels.vcf > /content/germline_indels_annotated.vcf

In [ ]:
import pandas as pd

def parse_vcf_to_df(vcf_path):
    variants = []
    with open(vcf_path, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            chunks = line.strip().split('\t')
            info = chunks[7]
            if "ANN=" in info:
                ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
                first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')
                gene_name = first_effect[3]
                effect = first_effect[1]
                impact = first_effect[2]
                variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])
    return pd.DataFrame(variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

# Generate individual dataframes
df_snps = parse_vcf_to_df("/content/germline_snps_annotated.vcf")
df_indels = parse_vcf_to_df("/content/germline_indels_annotated.vcf")

# --- DISPLAY RESULTS ---
print(f"=== GERMLINE SNPs DISCOVERED ({len(df_snps)} total) ===")
print(df_snps.head(10).to_string(index=False))

print("\n" + "="*60 + "\n")

print(f"=== GERMLINE INDELS DISCOVERED ({len(df_indels)} total) ===")
print(df_indels.head(10).to_string(index=False))

# Optional: Export them as clean spreadsheets!
df_snps.to_csv("/content/germline_snps_summary.csv", index=False)
df_indels.to_csv("/content/germline_indels_summary.csv", index=False)
print("\n💾 Saved summaries to 'germline_snps_summary.csv' and 'germline_indels_summary.csv'!")

## Conclusion and Key Findings

In this study, germline variant calling was performed using a standard GATK-based pipeline followed by functional annotation with SnpEff.

A total of **158 SNPs** and **17 INDELs** were identified from the input BAM-derived VCF files. The majority of variants were located in **non-coding regions**, including intronic and 3′ untranslated regions (3′ UTRs), and were classified as **MODIFIER or LOW impact**, suggesting limited predicted functional consequence.

Notably, most variants were mapped to genes such as *SLC5A12*, with a high proportion of synonymous and intronic substitutions, indicating strong evolutionary conservation of coding regions in this locus. A small number of INDELs showed **MODERATE impact**, including an in-frame insertion in *FAM83G*, which may warrant further functional investigation.

Overall, this dataset reflects a typical germline variant profile dominated by non-coding and low-impact changes. These results provide a foundation for downstream analyses such as population comparison, disease association studies, or integration with transcriptomic data.

The processed results were successfully exported as:
- `germline_snps_summary.csv`
- `germline_indels_summary.csv`